# Colab Full Run (Food101 + STL10)

This notebook clones the repo, installs deps, downloads Food101, runs extraction/training/eval/autointerp, and builds tables.


In [ ]:
# Repo clone
REPO_URL = "https://github.com/<USER>/<REPO>.git"  # TODO: set
REPO_DIR = "sae_for_clip"  # folder name in /content
REPO_PATH = f"/content/{REPO_DIR}"

import os

if not os.path.exists(REPO_PATH):
    !git clone {REPO_URL} {REPO_PATH}


In [ ]:
# Install deps
!pip install -r {REPO_PATH}/requirements.txt
!pip install -e {REPO_PATH}


In [ ]:
# Download Food101 via torchvision
from torchvision.datasets import Food101
Food101(root=f"{REPO_PATH}/data/torchvision", split="train", download=True)
FOOD_IMAGES = f"{REPO_PATH}/data/torchvision/food-101/images"


In [ ]:
# Extract activations
RUN_ACT = "colab_food101_acts"
!cd {REPO_PATH} && PYTHONPATH={REPO_PATH}/src python {REPO_PATH}/scripts/extract_activations.py \
  --image_dir {FOOD_IMAGES} \
  --model ViT-B-32 \
  --pretrained openai \
  --layer visual.transformer.resblocks.0 \
  --num_samples 50000 \
  --batch_size 128 \
  --shard_size 4096 \
  --out_dir {REPO_PATH}/artifacts/activation_cache \
  --run_name {RUN_ACT}


In [ ]:
# Train MSAE
RUN_SAE = "colab_msae"
!cd {REPO_PATH} && PYTHONPATH={REPO_PATH}/src python {REPO_PATH}/scripts/train_sae.py \
  --sae_type msae \
  --k_list 32,64,128,256 \
  --alpha_mode reverse \
  --input_centering dataset \
  --input_scaling dataset \
  --cache_dir {REPO_PATH}/artifacts/activation_cache/{RUN_ACT} \
  --dict_size 8192 \
  --epochs 10 \
  --batch_size 1024 \
  --lr 1e-3 \
  --l1_lambda 1e-4 \
  --device cuda \
  --run_name {RUN_SAE}


In [ ]:
# Zero-shot eval (CIFAR-10 + STL-10)
!cd {REPO_PATH} && PYTHONPATH={REPO_PATH}/src python {REPO_PATH}/scripts/eval_zeroshot.py --dataset cifar10 --split test --batch_size 256 --num_workers 2 --run_name c10_base
!cd {REPO_PATH} && PYTHONPATH={REPO_PATH}/src python {REPO_PATH}/scripts/eval_zeroshot.py --dataset stl10 --split test --batch_size 256 --num_workers 2 --run_name stl10_base

!cd {REPO_PATH} && PYTHONPATH={REPO_PATH}/src python {REPO_PATH}/scripts/eval_zeroshot.py \
  --dataset cifar10 --split test --batch_size 256 --num_workers 2 \
  --sae_checkpoint {REPO_PATH}/artifacts/checkpoints/{RUN_SAE}/last.pt \
  --sae_layer visual.transformer.resblocks.0 \
  --run_name c10_sae

!cd {REPO_PATH} && PYTHONPATH={REPO_PATH}/src python {REPO_PATH}/scripts/eval_zeroshot.py \
  --dataset stl10 --split test --batch_size 256 --num_workers 2 \
  --sae_checkpoint {REPO_PATH}/artifacts/checkpoints/{RUN_SAE}/last.pt \
  --sae_layer visual.transformer.resblocks.0 \
  --run_name stl10_sae

!cd {REPO_PATH} && PYTHONPATH={REPO_PATH}/src python {REPO_PATH}/scripts/make_p4_table.py \
  --eval_runs {REPO_PATH}/artifacts/eval/c10_sae {REPO_PATH}/artifacts/eval/stl10_sae \
  --checkpoint_dir {REPO_PATH}/artifacts/checkpoints/{RUN_SAE} \
  --out_path {REPO_PATH}/artifacts/eval/zeroshot_eval_table.md


In [ ]:
# Auto-interpretation (collages + CSV + table)
!cd {REPO_PATH} && PYTHONPATH={REPO_PATH}/src python {REPO_PATH}/scripts/build_collages.py \
  --cache_dir {REPO_PATH}/artifacts/activation_cache/{RUN_ACT} \
  --checkpoint {REPO_PATH}/artifacts/checkpoints/{RUN_SAE}/last.pt \
  --num_latents 300 --top_k 16 --image_size 128 \
  --out_dir {REPO_PATH}/artifacts/autointerp/collages \
  --manifest_path {REPO_PATH}/artifacts/autointerp/collages_manifest.csv

from getpass import getpass
import os
os.environ['OPENROUTER_API_KEY'] = getpass('OpenRouter API key: ')

!cd {REPO_PATH} && PYTHONPATH={REPO_PATH}/src python {REPO_PATH}/scripts/autointerp.py \
  --manifest_path {REPO_PATH}/artifacts/autointerp/collages_manifest.csv \
  --out_csv {REPO_PATH}/artifacts/autointerp/autointerp.csv

!cd {REPO_PATH} && PYTHONPATH={REPO_PATH}/src python {REPO_PATH}/scripts/make_p5_table.py \
  --autointerp_csv {REPO_PATH}/artifacts/autointerp/autointerp.csv \
  --out_path {REPO_PATH}/artifacts/autointerp/autointerp_table.md
